# 03 · Inference

Three ways to use the trained model: the pickled pipeline, the MLflow pyfunc flavour, and the REST API (local container or Kubernetes).

In [1]:
import sys, os
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)

In [2]:
import json
from src import config
from src.predict import load_model, predict_records
model = load_model()
patient = json.loads(Path('sample_request.json').read_text())
patient

{'age': 63,
 'sex': 1,
 'cp': 4,
 'trestbps': 150,
 'chol': 286,
 'fbs': 0,
 'restecg': 2,
 'thalach': 108,
 'exang': 1,
 'oldpeak': 1.5,
 'slope': 2,
 'ca': 3,
 'thal': 7}

## 1. Pickled scikit-learn pipeline (what the API loads)

In [3]:
predict_records(model, [patient])

[{'prediction': 1,
  'label': 'heart_disease',
  'probability_disease': 0.9978,
  'confidence': 0.9978,
  'risk_level': 'high'}]

In [4]:
healthy = {**patient, 'age': 41, 'sex': 0, 'cp': 2, 'thalach': 172, 'exang': 0, 'oldpeak': 0.0, 'slope': 1, 'ca': 0, 'thal': 3}
no_tests = {k: v for k, v in patient.items() if k not in ('ca', 'thal')}  # fluoroscopy / thallium not available
pd.DataFrame(predict_records(model, [patient, healthy, no_tests]), index=['high-risk', 'low-risk', 'missing ca/thal'])

,prediction,label,probability_disease,confidence,risk_level
high-risk,1,heart_disease,0.9978,0.9978,high
low-risk,0,no_heart_disease,0.0938,0.9062,low
missing ca/thal,1,heart_disease,0.8347,0.8347,high


## 2. MLflow model flavour (framework-agnostic `pyfunc`)

In [5]:
import mlflow.pyfunc
from src.predict import to_frame
pyfunc = mlflow.pyfunc.load_model(str(config.MLFLOW_MODEL_DIR))
pyfunc.predict(to_frame([patient, healthy]))

array([1, 0])

## 3. Batch scoring the held-out test set

In [6]:
from src.data import load_clean_data, split_features_target
from sklearn.model_selection import train_test_split
from src.evaluate import compute_metrics
X, y = split_features_target(load_clean_data())
_, X_test, _, y_test = train_test_split(X, y, test_size=config.TEST_SIZE, stratify=y, random_state=config.RANDOM_STATE)
proba = model.predict_proba(X_test)[:, 1]
{k: round(v, 3) for k, v in compute_metrics(y_test, (proba >= 0.5).astype(int), proba).items()}

{'accuracy': 0.869,
 'precision': 0.812,
 'recall': 0.929,
 'f1': 0.867,
 'roc_auc': 0.961,
 'pr_auc': 0.949}

## 4. REST API (`docker run -p 8000:8000 heart-disease-api:1.0.0` or the Kubernetes service)

In [7]:
import requests
API = os.getenv('HEART_API_URL', 'http://localhost:8000')
try:
    r = requests.post(f'{API}/predict', json=patient, timeout=3)
    print(r.status_code, r.headers.get('x-request-id')); print(json.dumps(r.json(), indent=2))
    bad = requests.post(f'{API}/predict', json={**patient, 'cp': 9}, timeout=3)
    print('invalid payload ->', bad.status_code, bad.json()['detail'][0]['msg'])
except requests.ConnectionError:
    print(f'API not running at {API} - start it with `make docker-run` or `make serve`')

200 845b988d7c8241e2
{
  "request_id": "845b988d7c8241e2",
  "prediction": 1,
  "label": "heart_disease",
  "confidence": 0.9978,
  "probability_disease": 0.9978,
  "risk_level": "high",
  "threshold": 0.5,
  "model_version": "1.0.0+logistic_regression"
}
invalid payload -> 422 Input should be 1, 2, 3 or 4
